# Player Statistics Data Preprocessing

This notebook performs the data preprocessing of scraped data for the model that predicts using player statistics. We merge and clean our raw data stored in `\data`.

The attributes I chose to use in building the model include:

**Points, Assists, Rebounds, Steals, Blocks, and Turnovers**, all on a per game basis. The model would have been improved by using statistics that further distinguish positions such as three point percentage (as guards and forwards are usually much better than centers in this department), turnovers or free throw percentage.  

However, due to these more advanced statistics rarely being recorded outside of professional games, I decided to only use the traditionally recorded statistics of points, assists, rebounds, steals and blocks as the average user who may have only played up to high school basketball would either have these attributes recorded or know a rough estimate of their numbers for these attributes. For example, american highschool varsity basketball only records these statistics: [See Here](https://www.maxpreps.com/basketball/stat-leaders/).

## Imports

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

## (1) Merge Data

In [93]:
# Retrieving relevant datasets
dates = ['20250728']
date_pattern = '|'.join(dates)
pattern = rf"nba_current_player_stats_(?:{date_pattern})_batch[0-9]+\.csv"

# Get list of datasets in \data which match the pattern
matching_files = [f for f in os.listdir("data") if re.match(pattern, f)]

print(matching_files)

# Read in and merge data
api_player_stats = pd.DataFrame()
for file in matching_files:
    df = pd.read_csv(os.path.join("data", file))
    api_player_stats = pd.concat([api_player_stats, df])

print(len(api_player_stats))
api_player_stats.head()

['nba_current_player_stats_20250728_batch1.csv', 'nba_current_player_stats_20250728_batch3.csv', 'nba_current_player_stats_20250728_batch2.csv']
2687


,player,pos,PTS,AST,REB,STL,BLK,TOV,AST_TO,STOCKS,FIC,age,year
0,Precious Achiuwa,F,4.98,0.48,3.41,0.33,0.46,0.70,0.685714,0.79,8.96,21.0,2021
1,Precious Achiuwa,F,9.10,1.12,6.48,0.51,0.56,1.15,0.973913,1.07,16.62,22.0,2022
2,Precious Achiuwa,F,9.24,0.91,5.96,0.56,0.55,1.07,0.850467,1.11,16.15,23.0,2023
3,Precious Achiuwa,F,7.72,1.76,5.44,0.64,0.48,1.16,1.517241,1.12,14.88,24.0,2024
4,Precious Achiuwa,F,7.59,1.08,7.16,0.61,1.14,1.10,0.981818,1.75,16.48,24.0,2024


## (2) Clean

In [94]:
print(f'The number of rows in the dataset is: {api_player_stats.shape[0]}')
print(f'The number of columns/features in the dataset is: {api_player_stats.shape[1]}')
print(f'The number of duplicate entries in the dataset is: {api_player_stats.duplicated().sum()}')
print(f'The number of missing values in the dataset is: {api_player_stats.isna().sum().sum()}')

The number of rows in the dataset is: 2687
The number of columns/features in the dataset is: 13
The number of duplicate entries in the dataset is: 0
The number of missing values in the dataset is: 0


Identify entries with `inf`. This happens with the AST/TO (Assist to Turnover) ratio when Turnover's is 0, resulting in a division by 0, creating infinity values.

In [95]:
print(api_player_stats[api_player_stats == np.inf].count())

player     0
pos        0
PTS        0
AST        0
REB        0
STL        0
BLK        0
TOV        0
AST_TO    20
STOCKS     0
FIC        0
age        0
year       0
dtype: int64


* There exists 20 records with an `inf` value for AST_TO.

Remove the 20 records with `inf`.

In [96]:
# Select only numeric columns
api_player_stats_numeric = api_player_stats.select_dtypes(include=[np.number])

# Find rows with any infinity values in numeric columns
api_player_stats[np.isinf(api_player_stats_numeric).any(axis=1)]

,player,pos,PTS,AST,REB,STL,BLK,TOV,AST_TO,STOCKS,FIC,age,year
266,Tony Bradley,C,0.89,0.11,1.22,0.00,0.00,0.0,inf,0.00,2.22,20.0,2018
438,Justin Champagnie,G,2.00,0.33,1.33,0.00,0.00,0.0,inf,0.00,3.66,22.0,2023
439,Justin Champagnie,G,2.50,1.50,2.00,0.50,0.00,0.0,inf,0.50,6.50,22.0,2023
440,Justin Champagnie,G,2.20,0.80,1.60,0.20,0.00,0.0,inf,0.20,4.80,22.0,2023
543,Seth Curry,G,0.00,0.50,1.00,0.00,0.00,0.0,inf,0.00,1.50,24.0,2015
642,PJ Dozier,G,3.17,0.83,2.83,0.33,0.00,0.0,inf,0.33,7.16,22.0,2019
750,Adam Flagler,G,1.50,2.00,0.00,0.00,0.00,0.0,inf,0.00,3.50,24.0,2024
817,Taj Gibson,F,4.50,0.50,2.25,0.25,0.25,0.0,inf,0.50,7.75,39.0,2024
973,Ron Harper Jr.,G,2.22,0.44,0.78,0.00,0.11,0.0,inf,0.11,3.55,23.0,2023
974,Ron Harper Jr.,G,0.00,1.00,0.00,0.00,0.00,0.0,inf,0.00,1.00,24.0,2024


In [97]:
# Remove these rows
api_player_stats = api_player_stats[~np.isinf(api_player_stats_numeric).any(axis=1)]

## (3) Export

In [ ]:
# Export api player stats
api_player_stats.to_csv(os.path.join("data", "api_player_stats.csv"), index=False)